# E-commerce Sales Analysis

End-to-end Python analysis of e-commerce transactions.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/raw/ecommerce_sales.csv')
df.head()

## 3. Data Quality Check

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 4. Data Cleaning

In [ ]:
df = df.drop_duplicates()
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['region'] = df['region'].fillna('Unknown')
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['discount_pct'] = df['discount_pct'].fillna(df['discount_pct'].median())
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0).astype(int)
df['month'] = df['order_date'].dt.to_period('M').astype(str)
df.to_csv('../data/processed/cleaned_sales.csv', index=False)
df.head()

## 5. Business KPIs

In [ ]:
completed = df[df['order_status'] == 'Completed'].copy()
total_revenue = completed['revenue'].sum()
total_orders = completed['order_id'].nunique()
aov = total_revenue / total_orders
print(f'Total Revenue: ₹{total_revenue:,.2f}')
print(f'Completed Orders: {total_orders:,}')
print(f'Average Order Value: ₹{aov:,.2f}')
print(f'Unique Customers: {completed["customer_id"].nunique():,}')

## 6. Monthly Revenue

In [ ]:
monthly = completed.groupby('month', as_index=False)['revenue'].sum()
monthly['growth_pct'] = monthly['revenue'].pct_change()*100
monthly

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly['month'], monthly['revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Category Performance

In [ ]:
category = completed.groupby('category', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique'),
    units=('quantity','sum')
).sort_values('revenue', ascending=False)
category

## 8. Top Products

In [ ]:
top_products = completed.groupby('product', as_index=False)['revenue'].sum().sort_values(
    'revenue', ascending=False
).head(10)
top_products

## 9. Regional Performance

In [ ]:
region = completed.groupby('region', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique')
).sort_values('revenue', ascending=False)
region

## 10. Business Questions

1. Which month generated the highest revenue?
2. Which category contributes the most revenue?
3. Which products are the top revenue contributors?
4. Which region has the highest sales?
5. What is the average order value?
6. What percentage of orders were cancelled or returned?

## 11. Conclusion

The analysis converts raw transaction data into KPIs and business insights. The workflow can be extended with SQL and Power BI.